# Test 7 — Manual Chunking + Manual Coref (American Civil War)

Third replication of the sentence-level coref-before-embed experiment (after Test 6 / French
Revolution), on a new entity-dense article. Chunking and coref were done **manually / via sub-agents**
with semantic judgment — no regex sentence splitter, no automated coref (LingMess/spaCy/neuralcoref).

**Claim under test:** *"small sentence chunks lose context."* A chunk like *"Its status had been
contentious for months"* carries no entity string; replacing the pronoun with the named entity before
embedding should make such chunks retrievable.

**Data (`test-7-data/`, ~5k words):**

| File | Role |
|------|------|
| `original_chunks.json` | 230 sentence chunks, pronouns intact (baseline) |
| `coref_chunks.json` | Same 230 chunks, manual coref (pronouns/demonstratives → named entities) |
| `eval_questions.json` | 30 questions, gold chunk ids, `coref_critical` flag |

**Honesty note on the questions:** queries are written as natural retrieval questions. Every query
content word appears somewhere in the **original** corpus (verified), so no question can be answered
*only* via a term introduced by the coref rewrite. `coref_critical=True` means the **gold** chunk hides
the queried entity behind a pronoun/demonstrative — it does **not** mean the baseline cannot retrieve
it (often it still can, via neighbouring context). Section 11 analyses what actually happened.

## Variants

| Label | Retrieval |
|-------|-----------|
| **baseline** | Dense on **original** sentence text |
| **coref_dense** | Dense on the **manual coref** rewrite |
| **coref_hybrid** | **Weighted RRF**: 0.7 × dense(coref) + 0.3 × BM25(original) |

## 1. Setup & Install

In [ ]:
!pip install -q pytrec_eval tqdm pandas numpy rank_bm25 sentence-transformers

## 2. Config

In [ ]:
import os, logging, glob
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("coref_v7")

EMBED_MODEL = "BAAI/bge-small-en-v1.5"
BATCH_SIZE  = 64
MAX_LEN     = 512

TOP_K        = 5
RRF_K        = 60
DENSE_WEIGHT = 0.7
RUN_DEPTH    = 100
CACHE_DIR    = "./embed_cache_v7"

def _resolve_data_dir():
    candidates = ["/kaggle/input/test-7-data", "/kaggle/input/test7-data", "./test-7-data", "test-7-data"]
    for c in candidates:
        if os.path.isdir(c) and glob.glob(os.path.join(c, "*chunks*.json")):
            return c
    for c in glob.glob("/kaggle/input/*"):
        if glob.glob(os.path.join(c, "*chunks*.json")):
            return c
    raise FileNotFoundError("Could not find test-7-data. Upload the 3 JSON files as a Kaggle Dataset "
                            "or place them in ./test-7-data/")

DATA_DIR = _resolve_data_dir()
os.makedirs(CACHE_DIR, exist_ok=True)
log.info("Config | model=%s | data_dir=%s | dense_w=%.2f | top_k=%d", EMBED_MODEL, DATA_DIR, DENSE_WEIGHT, TOP_K)

## 3. Load the three JSON files (1:1 chunk alignment)

In [ ]:
import json

def _load(name):
    with open(os.path.join(DATA_DIR, name), "r", encoding="utf-8") as f:
        return json.load(f)

def load_test7():
    original = _load("original_chunks.json")
    coref    = _load("coref_chunks.json")
    questions = _load("eval_questions.json")

    orig_by_id  = {c["chunk_id"]: c["text"] for c in original}
    coref_by_id = {c["chunk_id"]: c["text"] for c in coref}
    assert set(orig_by_id) == set(coref_by_id), "chunk id mismatch between original and coref"

    chunk_ids = sorted(orig_by_id)
    passages = [{"_id": str(cid), "text": orig_by_id[cid], "coref_text": coref_by_id[cid]} for cid in chunk_ids]

    queries, qrels, critical = {}, {}, {}
    for q in questions:
        qid = str(q["q_id"])
        queries[qid] = q["question"]
        qrels[qid] = {str(g): 1 for g in q["gold_chunk_ids"]}
        critical[qid] = bool(q.get("coref_critical", False))

    log.info("Loaded %d chunks | %d questions (%d coref-critical) | data_dir=%s",
             len(passages), len(queries), sum(critical.values()), DATA_DIR)
    return {"name": "acw_sentences", "passages": passages,
            "queries": queries, "qrels": qrels, "critical": critical}

ds = load_test7()
_ex = next(p for p in ds["passages"] if p["_id"] == "100")   # "Its status had been contentious..."
print("ORIG :", _ex["text"])
print("COREF:", _ex["coref_text"])

## 4. Encoder (bge-small dense) with disk cache

In [ ]:
import hashlib, pickle, re
import numpy as np
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

_has_cuda = False
try:
    import torch
    _has_cuda = torch.cuda.is_available()
except Exception:
    pass

PRONOUN_RE = re.compile(r"\b(he|she|him|her|his|hers|they|them|their|theirs|it|its)\b", re.IGNORECASE)

def _hash_texts(texts):
    h = hashlib.sha1()
    for t in texts:
        h.update(t.encode("utf-8", "ignore")); h.update(b"\x00")
    return h.hexdigest()[:16]

def _cache_file(dataset, variant, role, texts):
    return os.path.join(CACHE_DIR, f"{dataset}__{variant}__{role}__{_hash_texts(texts)}.pkl")

dense_model = SentenceTransformer(EMBED_MODEL, device="cuda" if _has_cuda else "cpu")
log.info("Loaded %s (dense) + rank_bm25 (lexical) | device=%s", EMBED_MODEL, "cuda" if _has_cuda else "cpu")

def encode_cached(dataset, variant, role, texts):
    cf = _cache_file(dataset, variant, role, texts)
    if os.path.exists(cf):
        with open(cf, "rb") as f:
            arr = pickle.load(f)
        log.info("cache HIT  | %s", os.path.basename(cf))
        return arr
    log.info("cache MISS | encoding %d %s texts (%s/%s)", len(texts), role, dataset, variant)
    arr = dense_model.encode(texts, batch_size=BATCH_SIZE, normalize_embeddings=True,
                             show_progress_bar=False).astype("float32")
    with open(cf, "wb") as f:
        pickle.dump(arr, f)
    return arr

## 5. Retrieval: dense, BM25, and weighted RRF

In [ ]:
def dense_ranklist(q_dense, p_dense, depth=RUN_DEPTH):
    sims = q_dense @ p_dense.T
    order = np.argsort(-sims, axis=1)[:, :depth]
    return [[(int(j), float(sims[qi, j])) for j in order[qi]] for qi in range(sims.shape[0])]

def bm25_ranklist(query_texts, passage_texts, depth=RUN_DEPTH):
    from rank_bm25 import BM25Okapi
    tok = lambda s: re.findall(r"[a-z0-9]+", s.lower())
    bm25 = BM25Okapi([tok(t) for t in passage_texts])
    out = []
    for q in tqdm(query_texts, desc="BM25", mininterval=2.0):
        scores = bm25.get_scores(tok(q))
        order = np.argsort(-scores)[:depth]
        out.append([(int(j), float(scores[j])) for j in order])
    return out

def weighted_rrf(dense_rl, bm25_rl, dense_w=DENSE_WEIGHT, k=RRF_K, depth=RUN_DEPTH):
    nq = len(dense_rl)
    w = {"dense": dense_w, "bm25": 1.0 - dense_w}
    fused = []
    for qi in range(nq):
        acc = {}
        for sysname, rl in (("dense", dense_rl), ("bm25", bm25_rl)):
            for rank, (pidx, _s) in enumerate(rl[qi]):
                acc[pidx] = acc.get(pidx, 0.0) + w[sysname] * 1.0 / (k + rank + 1)
        ordered = sorted(acc.items(), key=lambda x: -x[1])[:depth]
        fused.append([(int(p), float(s)) for p, s in ordered])
    return fused

## 6. Metrics (pytrec_eval) + coref-critical breakdown

In [ ]:
import pytrec_eval

def build_run(ranklist, query_ids, passage_ids):
    run = {}
    for qi, qid in enumerate(query_ids):
        run[qid] = {passage_ids[pidx]: float(score) for pidx, score in ranklist[qi]}
    return run

def eval_run(run, qrels, critical=None, k=TOP_K):
    ev = pytrec_eval.RelevanceEvaluator(qrels, {"recall_5", "ndcg_cut_10", "recip_rank"})
    per = ev.evaluate(run)
    recall = float(np.mean([v["recall_5"] for v in per.values()])) if per else 0.0
    ndcg = float(np.mean([v["ndcg_cut_10"] for v in per.values()])) if per else 0.0
    mrr = float(np.mean([v["recip_rank"] for v in per.values()])) if per else 0.0
    gold = {q: {d for d, s in rels.items() if s > 0} for q, rels in qrels.items()}
    hits = set()
    for qid, docs in run.items():
        topk = [d for d, _ in sorted(docs.items(), key=lambda x: -x[1])[:k]]
        if gold.get(qid) and (set(topk) & gold[qid]):
            hits.add(qid)
    out = {"recall_at_5": recall, "ndcg_at_10": ndcg, "mrr": mrr, "hits": hits, "n": len(qrels)}
    if critical is not None:
        crit_ids = [q for q in qrels if critical.get(q)]
        non_ids  = [q for q in qrels if not critical.get(q)]
        out["recall_crit"] = float(np.mean([per[q]["recall_5"] for q in crit_ids])) if crit_ids else 0.0
        out["recall_noncrit"] = float(np.mean([per[q]["recall_5"] for q in non_ids])) if non_ids else 0.0
        out["n_crit"] = len(crit_ids); out["n_noncrit"] = len(non_ids)
    return out

## 7. Run the three variants

In [ ]:
def run_dataset(ds):
    name = ds["name"]
    passages = ds["passages"]
    p_ids = [p["_id"] for p in passages]
    q_ids = list(ds["queries"].keys())
    q_texts = [ds["queries"][q] for q in q_ids]
    orig_texts = [p["text"] for p in passages]
    coref_texts = [p["coref_text"] for p in passages]

    q_dense       = encode_cached(name, "query", "query", q_texts)
    p_dense_orig  = encode_cached(name, "baseline", "passage", orig_texts)
    p_dense_coref = encode_cached(name, "coref_dense", "passage", coref_texts)

    rl_baseline = dense_ranklist(q_dense, p_dense_orig)
    rl_corefden = dense_ranklist(q_dense, p_dense_coref)
    rl_bm25     = bm25_ranklist(q_texts, orig_texts)
    rl_hybrid   = weighted_rrf(rl_corefden, rl_bm25)

    variants = {"baseline": rl_baseline, "coref_dense": rl_corefden, "coref_hybrid": rl_hybrid}
    results, runs, ranklists = {}, {}, {}
    for label, rl in variants.items():
        run = build_run(rl, q_ids, p_ids)
        runs[label] = run
        ranklists[label] = {q_ids[qi]: rl[qi] for qi in range(len(q_ids))}
        results[label] = eval_run(run, ds["qrels"], ds["critical"])
        r = results[label]
        log.info("[%s/%s] R@5=%.3f nDCG@10=%.3f MRR=%.3f | R@5(crit)=%.3f R@5(non)=%.3f",
                 name, label, r["recall_at_5"], r["ndcg_at_10"], r["mrr"],
                 r["recall_crit"], r["recall_noncrit"])
    return {"name": name, "results": results, "runs": runs, "ranklists": ranklists,
            "p_ids": p_ids, "n_passages": len(passages), "n_queries": len(q_ids)}

## 8. Results table + flip analysis

In [ ]:
import pandas as pd

def summarize(run_out):
    res = run_out["results"]; base = res["baseline"]
    rows = []
    for label in ["baseline", "coref_dense", "coref_hybrid"]:
        r = res[label]
        rows.append({"variant": label, "recall@5": round(r["recall_at_5"],4),
                     "nDCG@10": round(r["ndcg_at_10"],4), "MRR": round(r["mrr"],4),
                     "R@5_crit": round(r["recall_crit"],4), "R@5_noncrit": round(r["recall_noncrit"],4),
                     "Δrecall@5": round(r["recall_at_5"]-base["recall_at_5"],4),
                     "Δcrit": round(r["recall_crit"]-base["recall_crit"],4)})
    df = pd.DataFrame(rows)
    print(f"\n=== {run_out['name']} | passages={run_out['n_passages']} queries={run_out['n_queries']} ===")
    print(df.to_string(index=False))
    return df

def flips(run_out):
    res = run_out["results"]; base_hits = res["baseline"]["hits"]
    all_qids = set(run_out["runs"]["baseline"].keys()); out = {}
    for label in ["coref_dense", "coref_hybrid"]:
        h = res[label]["hits"]
        rec, hurt, bf = h - base_hits, base_hits - h, all_qids - base_hits - h
        out[label] = {"recovered": sorted(rec), "hurt": sorted(hurt), "both_fail": sorted(bf)}
        print(f"\n[{run_out['name']}] {label} vs baseline: recovered={len(rec)} hurt={len(hurt)} both_fail={len(bf)}")
        if rec:  print("   recovered q_ids:", sorted(rec, key=int))
        if hurt: print("   hurt q_ids     :", sorted(hurt, key=int))
    return out

run_out = run_dataset(ds)
df = summarize(run_out)
fl = flips(run_out)

## 9. Rank helper

Small utility to get the 1-based rank of a gold chunk within a variant's ranklist (or `None` if it
falls outside the retrieved depth). Used by the analysis section below.

In [ ]:
def gold_rank(run_out, variant, qid, gold_ids):
    """1-based rank of the best gold chunk in `variant` for `qid`; None if not in the run."""
    rl = run_out["ranklists"][variant][qid]           # list of (passage_index, score)
    p_ids = run_out["p_ids"]
    best = None
    for rank, (pidx, _s) in enumerate(rl, start=1):
        if p_ids[pidx] in gold_ids:
            best = rank if best is None else min(best, rank)
    return best

def pron_count(text):
    return len(PRONOUN_RE.findall(text))

## 10. Practical analysis — how did coreference actually perform?

This is the qualitative review of the mechanism, with real examples from this run:

1. **Corpus-level coref footprint** — how many chunks changed, and how many pronouns were removed.
2. **Every recovered query** — cases where `coref_dense` put the gold chunk in the top-5 and the
   baseline did not. For each we print the query, the gold chunk's original (pronoun) text vs the
   coref rewrite, and the gold rank under baseline → coref_dense → coref_hybrid.
3. **Any hurt query** — where coref pushed a previously-found gold chunk out of the top-5 (honest
   accounting of the downside).
4. **A coref-critical query that baseline already answered** — to show coref-critical does NOT mean
   "impossible without coref"; the dense model often finds pronoun-only chunks anyway.

In [ ]:
def analysis(ds, run_out, fl):
    by_id = {p["_id"]: p for p in ds["passages"]}
    passages = ds["passages"]

    # --- 1. corpus-level coref footprint ---
    changed = [p for p in passages if p["text"] != p["coref_text"]]
    pron_orig  = sum(pron_count(p["text"]) for p in passages)
    pron_coref = sum(pron_count(p["coref_text"]) for p in passages)
    print("=" * 78)
    print("1. COREF FOOTPRINT ON THE CORPUS")
    print("=" * 78)
    print(f"chunks total            : {len(passages)}")
    print(f"chunks changed by coref : {len(changed)} ({100*len(changed)/len(passages):.0f}%)")
    print(f"pronouns before coref   : {pron_orig}")
    print(f"pronouns after coref    : {pron_coref}  ({pron_orig - pron_coref} removed, "
          f"{100*(pron_orig-pron_coref)/max(pron_orig,1):.0f}% reduction)")
    print("(remaining pronouns are almost all inside preserved direct quotations)")

    # --- 2. recovered queries (the money shots) ---
    rec = fl["coref_dense"]["recovered"]
    print("\n" + "=" * 78)
    print(f"2. QUERIES RECOVERED BY coref_dense (baseline miss -> coref hit): {len(rec)}")
    print("=" * 78)
    for qid in sorted(rec, key=int):
        gold_ids = {pid for pid, s in ds["qrels"][qid].items() if s > 0}
        rb = gold_rank(run_out, "baseline", qid, gold_ids)
        rc = gold_rank(run_out, "coref_dense", qid, gold_ids)
        rh = gold_rank(run_out, "coref_hybrid", qid, gold_ids)
        print(f"\n[q{qid}] {ds['queries'][qid]}")
        print(f"       coref_critical = {ds['critical'].get(qid)}")
        for gid in sorted(gold_ids, key=int):
            p = by_id[gid]
            print(f"   gold chunk {gid}")
            print(f"     ORIGINAL (returned to user): {p['text']}")
            print(f"     COREF    (embedded)        : {p['coref_text']}")
            print(f"     pronouns  orig={pron_count(p['text'])}  coref={pron_count(p['coref_text'])}")
        print(f"   gold rank:  baseline={rb}  ->  coref_dense={rc}  |  coref_hybrid={rh}")
    if not rec:
        print("(none this run)")

    # --- 3. hurt queries (honest downside) ---
    hurt = fl["coref_dense"]["hurt"]
    print("\n" + "=" * 78)
    print(f"3. QUERIES HURT BY coref_dense (baseline hit -> coref miss): {len(hurt)}")
    print("=" * 78)
    for qid in sorted(hurt, key=int):
        gold_ids = {pid for pid, s in ds["qrels"][qid].items() if s > 0}
        rb = gold_rank(run_out, "baseline", qid, gold_ids)
        rc = gold_rank(run_out, "coref_dense", qid, gold_ids)
        print(f"\n[q{qid}] {ds['queries'][qid]}")
        for gid in sorted(gold_ids, key=int):
            p = by_id[gid]
            print(f"   gold {gid} ORIG : {p['text']}")
            print(f"   gold {gid} COREF: {p['coref_text']}")
        print(f"   gold rank:  baseline={rb}  ->  coref_dense={rc}  (fell out of top-{TOP_K})")
    if not hurt:
        print("(none this run — coref_dense hurt no queries)")

    # --- 4. a coref-critical query the baseline ALREADY answered ---
    print("\n" + "=" * 78)
    print("4. COREF-CRITICAL BUT BASELINE ALREADY FOUND IT (coref-critical != impossible)")
    print("=" * 78)
    base_hits = run_out["results"]["baseline"]["hits"]
    shown = 0
    for qid in sorted(ds["queries"], key=int):
        if not ds["critical"].get(qid) or qid not in base_hits:
            continue
        gold_ids = {pid for pid, s in ds["qrels"][qid].items() if s > 0}
        rb = gold_rank(run_out, "baseline", qid, gold_ids)
        rc = gold_rank(run_out, "coref_dense", qid, gold_ids)
        gid = sorted(gold_ids, key=int)[0]; p = by_id[gid]
        print(f"\n[q{qid}] {ds['queries'][qid]}")
        print(f"   gold {gid} ORIG : {p['text']}")
        print(f"   baseline still ranked gold at {rb}; coref_dense at {rc} "
              f"(neighbouring named context let dense find the pronoun-only chunk)")
        shown += 1
        if shown >= 3:
            break
    if shown == 0:
        print("(no coref-critical query was found by baseline this run)")

analysis(ds, run_out, fl)

## 11. Write `test-7-findings.md` (results section)

In [ ]:
def _md_table(df):
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "| " + " | ".join("---" for _ in cols) + " |"]
    for _, r in df.iterrows():
        lines.append("| " + " | ".join(str(r[c]) for c in cols) + " |")
    return "\n".join(lines)

def write_results(path="test-7-findings.md"):
    import time
    ro = run_out
    out = []
    out.append("\n\n---\n")
    out.append("## Measured Results (auto-generated)\n")
    out.append(f"**Run date:** {time.strftime('%Y-%m-%d')} | **Notebook:** `coref_public_eval_v7.ipynb`  ")
    out.append(f"**Models:** {EMBED_MODEL} (dense) + rank_bm25 (lexical) | "
               f"weighted-RRF {DENSE_WEIGHT:.1f}/{1-DENSE_WEIGHT:.1f} | TOP_K={TOP_K}\n")
    out.append(f"Chunks: {ro['n_passages']} (sentence-level, manually chunked) | Queries: {ro['n_queries']} "
               f"({ro['results']['baseline']['n_crit']} coref-critical, "
               f"{ro['results']['baseline']['n_noncrit']} non-critical)\n")
    out.append(_md_table(df) + "\n")
    base_hits = ro["results"]["baseline"]["hits"]; all_qids = set(ro["runs"]["baseline"].keys())
    out.append("**Flips vs baseline**\n")
    out.append("| variant | recovered | hurt | both_fail |")
    out.append("| --- | --- | --- | --- |")
    for label in ["coref_dense", "coref_hybrid"]:
        h = ro["results"][label]["hits"]
        out.append(f"| {label} | {len(h-base_hits)} | {len(base_hits-h)} | {len(all_qids-base_hits-h)} |")
    out.append("\n**How to read this:** `coref_critical` marks queries whose *gold* chunk hides the entity "
               "behind a pronoun. It does not mean the baseline cannot retrieve it. A rise in `R@5_crit` "
               "for `coref_dense` while `R@5_noncrit` stays flat is the signal that coref helped where it "
               "should. See notebook section 10 for the per-query recovered/hurt breakdown with examples.\n")
    text = "\n".join(out) + "\n"

    for target in [path, os.path.join(os.getcwd(), os.path.basename(path))]:
        try:
            mode = "a" if os.path.exists(target) else "w"
            with open(target, mode, encoding="utf-8") as f:
                f.write(text)
            log.info("Wrote results to %s (mode=%s, %d chars)", target, mode, len(text))
            break
        except Exception as e:
            log.warning("could not write %s: %s", target, e)
    print(text)

write_results()